In [ ]:
# =============================================================================
# Zelle 1: Umgebung, Bibliotheken, Pfade und Reproduzierbarkeit
# =============================================================================

#Bevor Sie das Notebook laufen lassen, installieren Sie bitte die benötigte Bibliotheken. 
#Der Prozess wurde in der README-Datei erläutert.

import os
import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import joblib

import tensorflow as tf
import sklearn

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import RidgeCV, TweedieRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline

import xgboost as xgb

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, SimpleRNN, GRU, Dense, Dropout, Bidirectional, BatchNormalization
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.optimizers import Adam

pd.set_option("display.max_columns", None)

import warnings
warnings.filterwarnings("ignore")

# Reproduzierbarkeit
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    print("Hinweis: Deterministische TensorFlow-Operationen konnten nicht vollständig aktiviert werden.")

print("TensorFlow-Version:", tf.__version__)
print("scikit-learn-Version:", sklearn.__version__)

# Projektpfade
# Erwartete Struktur:
# abgabe/
# ├── raw_data/
# └── code/
#     └── parkinson_thesis_reproducible.ipynb

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "code":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DATA_DIR = PROJECT_ROOT / "raw_data"
OUTPUT_DIR = PROJECT_ROOT / "code" / "outputs"

FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"
MODELS_DIR = OUTPUT_DIR / "models"
DASHBOARD_ASSETS_DIR = OUTPUT_DIR / "dashboard_assets"
DASHBOARD_ASSETS_FULL_DIR = OUTPUT_DIR / "dashboard_assets_full"

for directory in [
    OUTPUT_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    MODELS_DIR,
    DASHBOARD_ASSETS_DIR,
    DASHBOARD_ASSETS_FULL_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

DATA_FILES = {
    "train_proteins": RAW_DATA_DIR / "train_proteins.csv",
    "train_peptides": RAW_DATA_DIR / "train_peptides.csv",
    "train_clinical": RAW_DATA_DIR / "train_clinical_data.csv",
    "supplemental_clinical": RAW_DATA_DIR / "supplemental_clinical_data.csv",
}

missing_files = [str(path) for path in DATA_FILES.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Die folgenden Rohdatendateien fehlen im Ordner raw_data/:\n"
        + "\n".join(missing_files)
    )

print("Projektverzeichnis:", PROJECT_ROOT)
print("Rohdatenverzeichnis:", RAW_DATA_DIR)
print("Ausgabeverzeichnis:", OUTPUT_DIR)

In [ ]:
# =============================================================================
# Zelle 2: Laden der Rohdaten
# =============================================================================

print("Lade Rohdatensätze...")

train_proteins = pd.read_csv(DATA_FILES["train_proteins"])
train_peptides = pd.read_csv(DATA_FILES["train_peptides"])
train_clinical = pd.read_csv(DATA_FILES["train_clinical"])
supplemental_clinical = pd.read_csv(DATA_FILES["supplemental_clinical"])

clinical = pd.concat([train_clinical, supplemental_clinical], ignore_index=True)

clinical["upd23b_clinical_state_on_medication"] = (
    clinical["upd23b_clinical_state_on_medication"].fillna("Unknown")
)

print("Form der klinischen Daten:", clinical.shape)


In [ ]:
# =============================================================================
# Zelle 3: Vorverarbeitung der Protein- und Peptiddaten
# =============================================================================

chunk_size = 5000

print("Verarbeite Proteindaten...")
proteins_pivot = pd.DataFrame()

for chunk in pd.read_csv(DATA_FILES["train_proteins"], chunksize=chunk_size):
    chunk_pivot = (
        chunk.pivot_table(
            index="visit_id",
            columns="UniProt",
            values="NPX",
            aggfunc="mean"
        )
        .fillna(0)
    )
    proteins_pivot = (
        pd.concat([proteins_pivot, chunk_pivot], axis=0)
        .groupby(level=0)
        .mean()
    )

proteins_pivot = proteins_pivot.astype("float32")

print("Verarbeite Peptiddaten...")
peptides_pivot = pd.DataFrame()

for chunk in pd.read_csv(DATA_FILES["train_peptides"], chunksize=chunk_size):
    chunk_pivot = (
        chunk.pivot_table(
            index="visit_id",
            columns="Peptide",
            values="PeptideAbundance",
            aggfunc="mean"
        )
        .fillna(0)
    )
    peptides_pivot = (
        pd.concat([peptides_pivot, chunk_pivot], axis=0)
        .groupby(level=0)
        .mean()
    )

peptides_pivot = peptides_pivot.astype("float32")

print("Führe Protein- und Peptiddaten zusammen...")
bio_pivot = (
    proteins_pivot
    .merge(peptides_pivot, on="visit_id", how="outer")
    .fillna(0)
    .reset_index()
)

del proteins_pivot, peptides_pivot
gc.collect()

print("Form der biologischen Daten:", bio_pivot.shape)

In [ ]:
# =============================================================================
# Zelle 4: Feature Engineering (Ohne Data Leakage)
# =============================================================================

# 1. Zusammenführen von klinischen und biologischen Daten
merged = clinical.merge(bio_pivot, on='visit_id', how='left').fillna(0)
# Sortierung ist extrem wichtig für Zeitreihen-Features (Lags/Trends)
merged = merged.sort_values(['patient_id', 'visit_month'])

# 2. One-Hot-Encoding (Kategoriale Variable: Medikationsstatus umwandeln)
med_status_raw = merged['upd23b_clinical_state_on_medication'].copy()
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
med_encoded = encoder.fit_transform(merged[['upd23b_clinical_state_on_medication']])
med_df = pd.DataFrame(med_encoded, columns=encoder.get_feature_names_out())
merged = pd.concat([merged.drop('upd23b_clinical_state_on_medication', axis=1), med_df], axis=1)

# 3. SICHERE INTERPOLATION (Nur Forward Fill)
# Wir füllen Lücken mit dem letzten bekannten Wert auf, um zu verhindern, dass zukünftige
# Informationen in die Vergangenheit fließen (Zukunftssicherheit / No Leakage).
updrs_cols = ['updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']
for col in updrs_cols:
    merged[col] = merged.groupby('patient_id')[col].ffill().fillna(0)

# 4. SICHERE VERZÖGERUNGEN (Lags) UND DIFFERENZEN (Trends)
# Berechnung der Krankheitsdynamik anhand der letzten Besuche.
for col in updrs_cols:
    merged[f'{col}_lag1'] = merged.groupby('patient_id')[col].shift(1).fillna(0).astype('float32')
    merged[f'{col}_lag2'] = merged.groupby('patient_id')[col].shift(2).fillna(0).astype('float32')
    merged[f'{col}_trend'] = (merged[f'{col}_lag1'] - merged[f'{col}_lag2']).astype('float32') # Trend zwischen t-1 und t-2
    # Gleitender Durchschnitt der letzten 3 Perioden
    merged[f'{col}_roll_mean3'] = merged.groupby('patient_id')[col].shift(1).rolling(3, min_periods=1).mean().fillna(0).astype('float32')

# 5. Zyklische Zeit-Features
# Da Monate einen zyklischen Charakter haben (Monat 12 ist nah an Monat 1),
# kodieren wir dies mit Sinus- und Kosinus-Funktionen.
merged['visit_month_sin'] = np.sin(2 * np.pi * merged['visit_month'] / 12).astype('float32')
merged['visit_month_cos'] = np.cos(2 * np.pi * merged['visit_month'] / 12).astype('float32')

print("Feature Engineering abgeschlossen.")

In [ ]:
# =============================================================================
# Zelle 4.5: Erweiterte Klinische Analyse & Visualisierung (EDA)
# =============================================================================
print("--- ERWEITERTE KLINISCHE DATENANALYSE ---")

med_cols = [c for c in merged.columns if 'upd23b_clinical_state_on_medication' in c]
plot_df = merged.copy()
# Rekonstruktion des Medikationsstatus für die Visualisierung
plot_df['Medication_Status'] = plot_df[med_cols].idxmax(axis=1).apply(lambda x: x.split('_')[-1])

# Aggregation der Patientendaten für eine Übersicht
analysis_df = merged.groupby('patient_id').agg({
    'visit_month': 'count',                 # Gesamtanzahl der Besuche pro Patient
    'updrs_1': ['first', 'last', 'mean'],   # Erste, letzte und durchschnittliche UPDRS-1 Werte
    'updrs_2': ['first', 'last', 'mean'],
    'updrs_3': ['first', 'last', 'mean'],
    'updrs_4': ['first', 'last', 'mean']
})

# Multi-Index-Spalten abflachen
analysis_df.columns = ['_'.join(col).strip() for col in analysis_df.columns.values]
analysis_df = analysis_df.reset_index()

# Häufigster Medikationsstatus pro Patient
med_status_patient = plot_df.groupby('patient_id')['Medication_Status'].agg(lambda x: x.mode()[0] if not x.mode().empty else 'Unknown')
analysis_df = analysis_df.merge(med_status_patient, on='patient_id')

# Berechnung des Krankheitsprogressions-Scores (Differenz zwischen letztem und erstem Besuch)
analysis_df['Progression_Score'] = (analysis_df['updrs_1_last'] + analysis_df['updrs_2_last'] + analysis_df['updrs_3_last']) - \
                                   (analysis_df['updrs_1_first'] + analysis_df['updrs_2_first'] + analysis_df['updrs_3_first'])

# Einteilung der Patienten in Kategorien basierend auf der Besuchshäufigkeit
def categorize_visits(count):
    if count <= 3: return '0-3 Besuche'
    elif count <= 5: return '3-5 Besuche'
    elif count <= 8: return '5-8 Besuche'
    else: return '8+ Besuche'

analysis_df['Visit_Category'] = analysis_df['visit_month_count'].apply(categorize_visits)
visit_order = ['0-3 Besuche', '3-5 Besuche', '5-8 Besuche', '8+ Besuche']
analysis_df['Visit_Category'] = pd.Categorical(analysis_df['Visit_Category'], categories=visit_order, ordered=True)

# --- VISUALISIERUNGEN FÜR DIE THESIS ---
fig, axes = plt.subplots(2, 2, figsize=(20, 14))
plt.subplots_adjust(hspace=0.3)

# 1. Violin Plot: Einfluss der Medikation auf den Motor-Score
sns.violinplot(data=plot_df, x='Medication_Status', y='updrs_3', ax=axes[0, 0], palette='muted', inner='quartile')
axes[0, 0].set_title('Einfluss der Medikation auf den motorischen Score (UPDRS Teil 3)', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Medikationsstatus')
axes[0, 0].set_ylabel('Verteilung des motorischen Scores')

# 2. Korrelations-Scatterplot: Besuche vs. Progression
sns.regplot(data=analysis_df, x='visit_month_count', y='Progression_Score', ax=axes[0, 1],
            scatter_kws={'alpha':0.5}, line_kws={'color':'red'})
corr_val = analysis_df['visit_month_count'].corr(analysis_df['Progression_Score'])
axes[0, 1].set_title(f'Korrelation: Anzahl der Besuche vs. Krankheitsfortschritt (r={corr_val:.2f})', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Gesamtanzahl der Besuche')
axes[0, 1].set_ylabel('Krankheitsfortschritt (UPDRS-3 Differenz)')

# 3. Bar Plot: Durchschnittliche Scores nach Besuchshäufigkeit
summary_visits = analysis_df.groupby('Visit_Category')[['updrs_1_mean', 'updrs_2_mean', 'updrs_3_mean']].mean().reset_index()
summary_melted = summary_visits.melt(id_vars='Visit_Category', var_name='UPDRS_Part', value_name='Average_Score')

sns.barplot(data=summary_melted, x='Visit_Category', y='Average_Score', hue='UPDRS_Part', ax=axes[1, 0], palette='viridis')
axes[1, 0].set_title('Durchschnittliche UPDRS-Scores nach Besuchshäufigkeitsgruppe', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Besuchshäufigkeit')
axes[1, 0].set_ylabel('Durchschnittlicher UPDRS Score')
axes[1, 0].legend(title='UPDRS Teil')

# 4. Pie Chart: Verteilung der Patienten
visit_counts = analysis_df['Visit_Category'].value_counts().sort_index()
axes[1, 1].pie(visit_counts, labels=visit_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
axes[1, 1].set_title('Patientenverteilung nach Besuchshäufigkeit', fontsize=14, fontweight='bold')

plt.show()

print("\n--- ZUSAMMENFASSUNG: Durchschnittliche Scores nach Besuchshäufigkeit ---")
display_table = analysis_df.groupby('Visit_Category')[['updrs_1_mean', 'updrs_2_mean', 'updrs_3_mean', 'Progression_Score']].mean()
display_table.columns = ['Ø UPDRS 1', 'Ø UPDRS 2', 'Ø UPDRS 3', 'Ø Progression']
print(display_table)

In [ ]:
# =============================================================================
# Zelle 5: Erzeugung der Zielvariablen (Target Generation)
# =============================================================================
# Für die Vorhersage müssen wir die zukünftigen UPDRS-Werte als Zielvariablen festlegen.
# Wir sagen die Werte in +0, +6, +12 und +24 Monaten voraus.

offsets = [0, 6, 12, 24]
# Annahme: Ein Shift-Step entspricht ca. 3 Monaten.
shift_steps = [o // 3 for o in offsets]

target_cols = []
for offset, step in zip(offsets, shift_steps):
    for col in updrs_cols:
        target_col = f'{col}_plus_{offset}'
        # Zukünftige Werte der gleichen Person nach oben verschieben (shift neg.)
        merged[target_col] = merged.groupby('patient_id')[col].shift(-step)
        target_cols.append(target_col)

# Entfernen aller Zeilen, bei denen wir die Zukunft nicht kennen (z.B. der letzte Besuch des Patienten)
merged_final = merged.dropna(subset=target_cols).reset_index(drop=True)

drop_cols = target_cols + ['visit_id', 'patient_id', 'visit_month'] + updrs_cols
features = [c for c in merged_final.columns if c not in drop_cols]

# WICHTIG: Vor GroupKFold sicherstellen, dass die Daten streng sortiert sind,
# damit der Test-Split immer identisch und reproduzierbar bleibt.
merged_final = merged_final.sort_values(by=['patient_id', 'visit_month']).reset_index(drop=True)

unique_patients = merged_final['patient_id'].nunique()
total_samples = merged_final.shape[0]
total_features = merged_final.shape[1]
print(f"Finale Datenform: {merged_final.shape}")
print(f"-> Einzigartige Patienten: {unique_patients}")
print(f"-> Beobachtungen (Zeilen): {total_samples}")
print(f"-> Features (Spalten): {total_features}")
print("="*40 + "\n")

In [ ]:
# =============================================================================
# Zelle 6: Datenaufteilung (Split) und Skalierung
# =============================================================================

X = merged_final[features]
y = merged_final[target_cols]
groups = merged_final['patient_id'] # Gruppen für GroupKFold

# GroupKFold verhindert, dass Daten desselben Patienten sowohl im Trainings-
# als auch im Testset landen. Dies ist extrem wichtig in der medizinischen ML!
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X, y, groups))

X_train_raw, X_test_raw = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Standardisierung der Eingabemerkmale (Mittelwert 0, Standardabweichung 1)
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train_raw), columns=features, index=X_train_raw.index)
X_test = pd.DataFrame(scaler.transform(X_test_raw), columns=features, index=X_test_raw.index)

all_predictions = {}
print("Datenvorbereitung abgeschlossen.")

In [ ]:
# =============================================================================
# Zelle 6.5: Hyperparameter Tuning (Optuna - Optimiert & Gecacht)
# =============================================================================
RUN_OPTUNA = False # Auf False gesetzt, um Notebook-Laufzeiten zu sparen (Werte sind bereits optimiert)

if RUN_OPTUNA:
    print("Starte Hyperparameter Tuning (Optuna)... Dies kann dauern.")

    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'objective': 'reg:squarederror',
            'n_jobs': -1,
            'random_state': 42
        }
        target_sample = y_train.iloc[:, 0]
        model = xgb.XGBRegressor(**params)

        # Cross-Validation innerhalb der Studie
        gkf_opt = GroupKFold(n_splits=3)
        scores = []
        groups_train = groups.iloc[train_idx]

        for tr_idx, val_idx in gkf_opt.split(X_train, target_sample, groups_train):
            X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
            y_tr, y_val = target_sample.iloc[tr_idx], target_sample.iloc[val_idx]

            model.fit(X_tr, y_tr)
            preds = model.predict(X_val)
            scores.append(mean_absolute_error(y_val, preds))

        return np.mean(scores)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=10)
    best_xgb_params = study.best_params
    print("Neue beste Parameter gefunden:", best_xgb_params)

else:
    print("Optuna wird übersprungen. Vorberechnete BESTE Parameter werden verwendet.")
    # Dies sind die optimierten Parameter aus vorherigen Läufen (Vermeidung langer Trainingszeiten für die Thesis-Korrektur)
    best_xgb_params = {
        'n_estimators': 411,
        'max_depth': 3,
        'learning_rate': 0.045674755828728035,
        'subsample': 0.9223030571641937,
        'colsample_bytree': 0.806414076713162,
        'n_jobs': -1,
        'random_state': 42
    }
    print("Geladene Parameter:", best_xgb_params)

In [ ]:
# =============================================================================
# Zelle 7: Lineare Modelle (Ridge & Generalized Linear Model)
# =============================================================================
print("Trainiere Lineare Modelle...")

# 1. Ridge Regression (L2-Regularisierung zur Vermeidung von Overfitting)
ridge = MultiOutputRegressor(RidgeCV(alphas=[0.1, 1.0, 10.0]))
ridge.fit(X_train, y_train)
all_predictions['Lineares Modell (Ridge)'] = ridge.predict(X_test)

# 2. GLM (Generalized Linear Model - Tweedie Regressor)
# Tweedie power=1 (Poisson), power=2 (Gamma).
# Da UPDRS-Daten oft bei Null zentriert sind und eine rechtsschiefe Verteilung haben,
# ist power=1.5 (Compound Poisson-Gamma) eine statistisch robuste Wahl.
print("Trainiere GLM (Tweedie Regressor)...")
glm = MultiOutputRegressor(TweedieRegressor(power=1.5, alpha=0.5, max_iter=1000))
glm.fit(X_train, y_train)
all_predictions['GLM'] = glm.predict(X_test)

print("Lineare Modelle & GLM Abgeschlossen.")

In [ ]:
# =============================================================================
# Zelle 8: XGBoost
# =============================================================================
print("Trainiere XGBoost mit optimierten Parametern...")

final_xgb_params = best_xgb_params.copy()
final_xgb_params['objective'] = 'reg:squarederror'
final_xgb_params['n_jobs'] = -1

xgb_model = xgb.XGBRegressor(**final_xgb_params)
xgb_model.fit(X_train, y_train)

all_predictions['XGBoost'] = xgb_model.predict(X_test)
print("XGBoost Abgeschlossen.")

In [ ]:
# =============================================================================
# Zelle 9: Random Forest
# =============================================================================
print("Trainiere Random Forest...")

rf_model = RandomForestRegressor(n_estimators=150, max_depth=12, n_jobs=-1, random_state=42)
rf_model.fit(X_train, y_train)

all_predictions['Random Forest'] = rf_model.predict(X_test)
print("Random Forest Abgeschlossen.")

In [ ]:
# =============================================================================
# Zelle 10: Support Vector Regression (SVR)
# =============================================================================
print("Trainiere Support Vector Regression (SVR)...")

# SVR reagiert empfindlich auf zu viele Dimensionen (Curse of Dimensionality).
# Daher wenden wir zuerst PCA (Principal Component Analysis) an, um die Features zu reduzieren.
svr_pipeline = Pipeline([
    ('pca', PCA(n_components=30)),
    ('svr', SVR(kernel='rbf', C=1.0, epsilon=0.1))
])

# MultiOutputRegressor kapselt das SVR-Modell für unsere 16 Ziele.
svr_wrapper = MultiOutputRegressor(svr_pipeline, n_jobs=1)   # n_jobs=1 ist wichtig für Stabilität

# Kopie der Daten, um "writeable flag"-Fehler zu vermeiden
X_train_svr = X_train.copy()
X_test_svr = X_test.copy()

svr_wrapper.fit(X_train_svr, y_train)
all_predictions['SVR'] = svr_wrapper.predict(X_test_svr)

print("SVR Abgeschlossen.")

In [ ]:
# =============================================================================
# Zelle 11: Deep Learning (LSTM, RNN, GRU) - AKADEMISCHES NIVEAU
# =============================================================================
from tensorflow.keras.optimizers import Adam

# Skalierung der Zielvariablen (Y) ist wichtig für die Stabilität und
# Konvergenz von Deep Learning Modellen (verhindert Exploding Gradients).
y_scaler = StandardScaler()
y_train_scaled_arr = y_scaler.fit_transform(y_train)
y_train_scaled = pd.DataFrame(y_train_scaled_arr, columns=y_train.columns, index=y_train.index)

# 🚨 KORRIGIERTE, THESIS-TAUGLICHE FUNKTION ZUR SEQUENZERSTELLUNG 🚨
def create_patient_sequences(X_df, y_df, patient_groups, seq_len=3):
    """
    Erstellt Sequenzen für rekurrente neuronale Netze (RNN).
    Verhindert Datenlecks (Data Leakage) zwischen Patienten.
    Wendet Zero-Padding an für die ersten Besuche eines Patienten (wenn keine Historie existiert).
    Dadurch bleibt die Zeilenanzahl 1:1 identisch mit den ML-Modellen (z.B. XGBoost).
    """
    X_seq, y_seq = [], []

    X_arr = X_df.values
    y_arr = y_df.values
    p_arr = patient_groups.values

    for idx in range(len(X_df)):
        current_patient = p_arr[idx]
        patient_history = []

        # Rückblickende Suche (t-2, t-1, t)
        for step in range(seq_len - 1, -1, -1):
            prev_idx = idx - step
            # Überprüfen, ob der Index gültig ist UND demselben Patienten gehört
            if prev_idx >= 0 and p_arr[prev_idx] == current_patient:
                patient_history.append(X_arr[prev_idx])
            else:
                # Falls keine Historie vorhanden (z.B. erster Besuch), fülle mit Nullen (Zero-Padding)
                patient_history.append(np.zeros(X_arr.shape[1]))

        X_seq.append(patient_history)
        y_seq.append(y_arr[idx])

    return np.array(X_seq), np.array(y_seq)

seq_len = 3
print("Bereite DL-Sequenzen mit Zero-Padding vor (Kein Data Leakage)...")

groups_train = groups.loc[X_train.index]
groups_test = groups.loc[X_test.index]

X_train_seq, y_train_seq_scaled = create_patient_sequences(X_train, y_train_scaled, groups_train, seq_len)
X_test_seq, y_test_seq = create_patient_sequences(X_test, y_test, groups_test, seq_len)

print(f"Form X_train_seq: {X_train_seq.shape} (Stimmt mit X_train überein: {X_train.shape[0]})")
print(f"Form X_test_seq: {X_test_seq.shape} (Stimmt mit X_test überein: {X_test.shape[0]})")

callbacks = [ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=0),
             EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)]

adam_opt = lambda: Adam(learning_rate=0.001)

# 1. Long Short-Term Memory (LSTM)
print("Trainiere LSTM...")
lstm_model = Sequential([
    Bidirectional(LSTM(64, return_sequences=True), input_shape=(seq_len, X_train.shape[1])),
    BatchNormalization(), Dropout(0.3),
    Bidirectional(LSTM(32)),
    BatchNormalization(), Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(y_train.shape[1])
])
lstm_model.compile(optimizer=adam_opt(), loss='mae')
lstm_model.fit(X_train_seq, y_train_seq_scaled, epochs=50, batch_size=64, validation_split=0.1, callbacks=callbacks, verbose=0)
pred_lstm = y_scaler.inverse_transform(lstm_model.predict(X_test_seq)) # Rücktransformation!

# 2. Recurrent Neural Network (SimpleRNN)
print("Trainiere SimpleRNN...")
rnn_model = Sequential([
    SimpleRNN(64, return_sequences=True, input_shape=(seq_len, X_train.shape[1])),
    BatchNormalization(), Dropout(0.3),
    SimpleRNN(32),
    BatchNormalization(), Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(y_train.shape[1])
])
rnn_model.compile(optimizer=adam_opt(), loss='mae')
rnn_model.fit(X_train_seq, y_train_seq_scaled, epochs=50, batch_size=64, validation_split=0.1, callbacks=callbacks, verbose=0)
pred_rnn = y_scaler.inverse_transform(rnn_model.predict(X_test_seq))

# 3. Gated Recurrent Unit (GRU)
print("Trainiere GRU...")
gru_model = Sequential([
    Bidirectional(GRU(64, return_sequences=True), input_shape=(seq_len, X_train.shape[1])),
    BatchNormalization(), Dropout(0.3),
    Bidirectional(GRU(32)),
    BatchNormalization(), Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(y_train.shape[1])
])
gru_model.compile(optimizer=adam_opt(), loss='mae')
gru_model.fit(X_train_seq, y_train_seq_scaled, epochs=50, batch_size=64, validation_split=0.1, callbacks=callbacks, verbose=0)
pred_gru = y_scaler.inverse_transform(gru_model.predict(X_test_seq))

In [ ]:
# =============================================================================
# Zelle 12: Ensemble Modell
# =============================================================================
# Ensemble Learning kombiniert die Stärken verschiedener Modelle.
# Durch manuelle Zuweisung von Gewichten erhalten leistungsstärkere Modelle (wie XGBoost) mehr Einfluss.

pred_xgb_c = all_predictions['XGBoost']
pred_rf_c = all_predictions['Random Forest']
pred_ridge_c = all_predictions['Lineares Modell (Ridge)']
pred_glm_c = all_predictions['GLM']
pred_svr_c = all_predictions['SVR']

# Gewichte (Weights) basierend auf bisherigen Performance-Beobachtungen
w_xgb, w_rf = 0.25, 0.20
w_ridge, w_glm, w_svr = 0.05, 0.05, 0.10
w_lstm, w_rnn, w_gru = 0.15, 0.05, 0.15

pred_ensemble = (w_xgb * pred_xgb_c) + (w_rf * pred_rf_c) + \
                (w_ridge * pred_ridge_c) + (w_glm * pred_glm_c) + (w_svr * pred_svr_c) + \
                (w_lstm * pred_lstm) + (w_rnn * pred_rnn) + (w_gru * pred_gru)

final_preds = {
    'XGBoost': pred_xgb_c, 'Random Forest': pred_rf_c,
    'Ridge': pred_ridge_c, 'GLM': pred_glm_c, 'SVR': pred_svr_c,
    'LSTM': pred_lstm, 'RNN': pred_rnn, 'GRU': pred_gru,
    'Ensemble': pred_ensemble
}

y_test_trimmed = y_test # Das Testset bleibt nun in Originalgröße dank Zero-Padding.

print("Ensemble Modell erfolgreich berechnet!")

In [ ]:
# =============================================================================
# Zelle 13: Detaillierte Metriken und Heatmap
# =============================================================================

results = []
true_vals_df = y_test_trimmed.reset_index(drop=True)

for model_name, preds in final_preds.items():
    for i, col in enumerate(target_cols):
        true_vals = true_vals_df[col].values
        pred_vals = preds[:, i]

        mae = mean_absolute_error(true_vals, pred_vals)
        rmse = np.sqrt(mean_squared_error(true_vals, pred_vals))

        numerator = np.abs(true_vals - pred_vals)
        denominator = (np.abs(true_vals) + np.abs(pred_vals)) / 2
        smape = 100 * np.mean(numerator / (denominator + 1e-8))

        results.append({
            "Modell": model_name,
            "Zielvariable": col,
            "MAE": mae,
            "RMSE": rmse,
            "SMAPE": smape
        })

metrics_df = pd.DataFrame(results)
metrics_path = TABLES_DIR / "finale_metriken.csv"
metrics_df.to_csv(metrics_path, index=False)

pivot_mae = metrics_df.pivot(index="Modell", columns="Zielvariable", values="MAE")

plt.figure(figsize=(18, 10))
sns.heatmap(pivot_mae, annot=True, fmt=".2f", cmap="viridis", linewidths=.5)
plt.title("Finale Modellleistung auf Basis des Mean Absolute Error", fontsize=16)
plt.tight_layout()

heatmap_path = FIGURES_DIR / "finale_modellleistung_heatmap.png"
plt.savefig(heatmap_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Metriktabelle gespeichert unter: {metrics_path}")
print(f"Heatmap gespeichert unter: {heatmap_path}")

In [ ]:
# =============================================================================
# Zelle 14: Vergleichstabelle der Modellvorhersagen
# =============================================================================

test_ids = groups.iloc[test_idx].values
test_months = merged_final.iloc[test_idx]["visit_month"].values

df_comp = pd.DataFrame({
    "patient_id": test_ids,
    "visit_month": test_months
})

y_test_reset = y_test.reset_index(drop=True)

for col in target_cols:
    df_comp[f"Wahr_{col}"] = y_test_reset[col]

for model_name, preds in final_preds.items():
    temp_df = pd.DataFrame(
        preds,
        columns=[f"{model_name}_{c}" for c in target_cols]
    )
    df_comp = pd.concat(
        [df_comp.reset_index(drop=True), temp_df.reset_index(drop=True)],
        axis=1
    )

comparison_path = TABLES_DIR / "vorhersagen_vergleich.csv"
df_comp.to_csv(comparison_path, index=False)

print(f"Vergleichstabelle gespeichert unter: {comparison_path}")

In [ ]:
# =============================================================================
# Zelle 15: Visualisierung der Vorhersagen auf Patientenebene
# =============================================================================
def plot_patient(patient_id, df_data):
    """
    Zeichnet die wahren Werte (True) gegen die Vorhersagen aller Modelle
    für einen spezifischen Patienten (Wichtig für das Dashboard / Fallstudien in der Thesis).
    """
    p_data = df_data[df_data['patient_id'] == patient_id]

    if len(p_data) == 0:
        print(f"Keine Daten gefunden für Patient ID: {patient_id}")
        return

    visit_row = p_data.iloc[0]
    visit_month = int(visit_row['visit_month'])

    print(f"Visualisiere Patient ID: {patient_id}, Besuchsmonat: {visit_month}")

    models = [
        'Wahr',
        'XGBoost', 'Random Forest', 'Ridge', 'GLM', 'SVR',
        'LSTM', 'RNN', 'GRU', 'Ensemble'
    ]
    horizons = [0, 6, 12, 24]

    fig, axes = plt.subplots(2, 2, figsize=(24, 14))
    axes = axes.flatten()

    for i, updrs_part in enumerate([1, 2, 3, 4]):
        ax = axes[i]
        plot_data = {'Horizont': [], 'Score': [], 'Quelle': []}

        for horizon in horizons:
            col_suffix = f'updrs_{updrs_part}_plus_{horizon}'

            # Wahre Werte
            true_col = f'Wahr_{col_suffix}'
            if true_col in visit_row:
                plot_data['Horizont'].append(f'+{horizon}m')
                plot_data['Score'].append(visit_row[true_col])
                plot_data['Quelle'].append('Wahr')

            # Vorhersagen
            for model in models[1:]:
                pred_col = f'{model}_{col_suffix}'
                if pred_col in visit_row:
                    val = visit_row[pred_col]
                    if not pd.isna(val):
                        plot_data['Horizont'].append(f'+{horizon}m')
                        plot_data['Score'].append(val)
                        plot_data['Quelle'].append(model)

        if plot_data['Score']:
            pdf = pd.DataFrame(plot_data)
            sns.barplot(data=pdf, x='Horizont', y='Score', hue='Quelle', ax=ax, palette='bright', alpha=0.9)

        ax.set_title(f'UPDRS Teil {updrs_part} Vorhersagen', fontsize=14, fontweight='bold')
        ax.set_xlabel('Vorhersagehorizont')
        ax.set_ylabel('UPDRS Score')
        ax.grid(True, axis='y', linestyle='--', alpha=0.5)

        if i == 0:
            ax.legend(loc='lower left', bbox_to_anchor=(0, 1.10, 1, 0.2), mode="expand", borderaxespad=0, ncol=6)
        else:
            if ax.get_legend(): ax.get_legend().remove()

    plt.suptitle(f'Vollständiger Modellvergleich für Patient {patient_id}', fontsize=18, y=1.05)
    plt.tight_layout()

    filename = f'patient_{patient_id}_analyse_alle_modelle.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Gespeichert: {filename}")
    plt.show()

# Ausführung der Patienten-Visualisierung
if 'df_comp' in globals():
    if 51708 in df_comp['patient_id'].values:
        plot_patient(51708, df_comp)
    else:
        p_id = np.random.choice(df_comp['patient_id'].unique())
        plot_patient(p_id, df_comp)
else:
    print("FEHLER: Die Tabelle 'df_comp' wurde nicht gefunden. Bitte führe Zelle 14 zuerst aus.")

In [ ]:
# =============================================================================
# Zelle 16: Feature Importance (Merkmalsbedeutung) Analyse
# =============================================================================
def plot_imp(models, features, top_n=15):
    """
    Zeigt, welche biologischen und klinischen Merkmale vom Modell am stärksten
    für die Vorhersagen gewichtet werden (Tree-based Feature Importance).
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    # XGBoost
    xgb_imp = pd.DataFrame({'Merkmal': features, 'Wichtigkeit': models['XGBoost'].feature_importances_})
    sns.barplot(data=xgb_imp.sort_values('Wichtigkeit', ascending=False).head(top_n), x='Wichtigkeit', y='Merkmal', ax=axes[0], palette='viridis')
    axes[0].set_title('XGBoost Merkmalsbedeutung')

    # Random Forest
    rf_imp = pd.DataFrame({'Merkmal': features, 'Wichtigkeit': models['Random Forest'].feature_importances_})
    sns.barplot(data=rf_imp.sort_values('Wichtigkeit', ascending=False).head(top_n), x='Wichtigkeit', y='Merkmal', ax=axes[1], palette='magma')
    axes[1].set_title('Random Forest Merkmalsbedeutung')

    plt.tight_layout()
    filename = 'feature_importance_original.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Gespeichert: {filename}")
    plt.show()

current_models = {'XGBoost': xgb_model, 'Random Forest': rf_model}
plot_imp(current_models, features)

In [ ]:
# Cell 17: Top-Feature Selection, Fixed Params, Retraining & Heatmap Save

print("--- SCHRITT 1: AUSWAHL DER TOP 50 BIO-FEATURES ---")

# 1. FEATURE-AUSWAHL
feature_imp = pd.DataFrame({'Feature': features, 'Importance': xgb_model.feature_importances_})
clinical_keywords = ['lag', 'trend', 'roll', 'visit_month', 'medication']
clinical_feats = [f for f in features if any(k in f for k in clinical_keywords)]
bio_feats_all = [f for f in features if f not in clinical_feats]
top_50_bio_df = feature_imp[feature_imp['Feature'].isin(bio_feats_all)].sort_values('Importance', ascending=False).head(50)
top_50_bio_feats = top_50_bio_df['Feature'].tolist()

selected_features = clinical_feats + top_50_bio_feats
print(f"Ausgewählte Feature-Anzahl: {len(selected_features)} ({len(clinical_feats)} klinisch + 50 Bio)")

X_train_opt = X_train[selected_features]
X_test_opt = X_test[selected_features]

# --- SCHRITT 2: FIXED OPTIMIZED PARAMETERS (NO OPTUNA) ---
print("\n--- SCHRITT 2: VERWENDE FESTE OPTIMIERTE PARAMETER ---")

xgb_opt_params = {
    'n_estimators': 473,
    'max_depth': 3,
    'learning_rate': 0.017,
    'subsample': 0.65,
    'colsample_bytree': 0.76,
    'objective': 'reg:squarederror',
    'n_jobs': -1,
    'random_state': 42
}

print("Verwendete Parameter:", xgb_opt_params)

# --- SCHRITT 3: RETRAINING ALLER MODELLE ---
print("\n--- SCHRITT 3: RETRAINING DER MODELLE MIT NEUEN FEATURES ---")
all_preds_opt = {}

# Ridge
ridge_opt = MultiOutputRegressor(RidgeCV(alphas=[0.1, 1.0, 10.0, 50.0]))
ridge_opt.fit(X_train_opt, y_train)
all_preds_opt['Ridge'] = ridge_opt.predict(X_test_opt)

# GLM
glm_opt = MultiOutputRegressor(TweedieRegressor(power=1.5, alpha=0.5, max_iter=1000))
glm_opt.fit(X_train_opt, y_train)
all_preds_opt['GLM'] = glm_opt.predict(X_test_opt)

# XGBoost (FIXED PARAMS)
xgb_opt = xgb.XGBRegressor(**xgb_opt_params)
xgb_opt.fit(X_train_opt, y_train)
all_preds_opt['XGBoost'] = xgb_opt.predict(X_test_opt)

# Random Forest
rf_opt = RandomForestRegressor(n_estimators=200, max_depth=15, n_jobs=-1, random_state=42)
rf_opt.fit(X_train_opt, y_train)
all_preds_opt['Random Forest'] = rf_opt.predict(X_test_opt)

# SVR
svr_opt_pipeline = Pipeline([
    ('pca', PCA(n_components=min(30, X_train_opt.shape[1]))),
    ('svr', SVR(kernel='rbf', C=1.0, epsilon=0.1))
])
svr_opt = MultiOutputRegressor(svr_opt_pipeline, n_jobs=1)
svr_opt.fit(X_train_opt.copy(), y_train)
all_preds_opt['SVR'] = svr_opt.predict(X_test_opt.copy())

# --- SCHRITT 4: DEEP LEARNING ---
print("\n--- SCHRITT 4: DEEP LEARNING (MIT TARGET-SKALIERUNG) ---")

y_scaler_opt = StandardScaler()
y_train_scaled = y_scaler_opt.fit_transform(y_train)
y_train_scaled = pd.DataFrame(y_train_scaled, columns=y_train.columns, index=y_train.index)

seq_len = 3
groups_train_opt = groups.loc[X_train_opt.index]
groups_test_opt = groups.loc[X_test_opt.index]

X_train_seq_opt, y_train_seq_scaled = create_patient_sequences(X_train_opt, y_train_scaled, groups_train_opt, seq_len)
X_test_seq_opt, _ = create_patient_sequences(X_test_opt, y_test, groups_test_opt, seq_len)

callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=0),
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
]

def build_dl_model(model_type, input_shape, output_shape):
    model = Sequential()
    if model_type == 'LSTM':
        model.add(Bidirectional(LSTM(128, return_sequences=True), input_shape=input_shape))
        model.add(BatchNormalization()); model.add(Dropout(0.3))
        model.add(Bidirectional(LSTM(64)))
    elif model_type == 'RNN':
        model.add(SimpleRNN(128, return_sequences=True, input_shape=input_shape))
        model.add(BatchNormalization()); model.add(Dropout(0.3))
        model.add(SimpleRNN(64))
    elif model_type == 'GRU':
        model.add(Bidirectional(GRU(128, return_sequences=True), input_shape=input_shape))
        model.add(BatchNormalization()); model.add(Dropout(0.3))
        model.add(Bidirectional(GRU(64)))

    model.add(BatchNormalization()); model.add(Dropout(0.3))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(output_shape))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mae')
    return model

print("Training LSTM...")
lstm_opt = build_dl_model('LSTM', (seq_len, X_train_opt.shape[1]), y_train.shape[1])
lstm_opt.fit(X_train_seq_opt, y_train_seq_scaled, epochs=50, batch_size=64, validation_split=0.1, callbacks=callbacks, verbose=0)
pred_lstm_opt = y_scaler_opt.inverse_transform(lstm_opt.predict(X_test_seq_opt))

print("Training RNN...")
rnn_opt = build_dl_model('RNN', (seq_len, X_train_opt.shape[1]), y_train.shape[1])
rnn_opt.fit(X_train_seq_opt, y_train_seq_scaled, epochs=50, batch_size=64, validation_split=0.1, callbacks=callbacks, verbose=0)
pred_rnn_opt = y_scaler_opt.inverse_transform(rnn_opt.predict(X_test_seq_opt))

print("Training GRU...")
gru_opt = build_dl_model('GRU', (seq_len, X_train_opt.shape[1]), y_train.shape[1])
gru_opt.fit(X_train_seq_opt, y_train_seq_scaled, epochs=50, batch_size=64, validation_split=0.1, callbacks=callbacks, verbose=0)
pred_gru_opt = y_scaler_opt.inverse_transform(gru_opt.predict(X_test_seq_opt))

# --- SCHRITT 5: ENSEMBLE ---
print("\n--- SCHRITT 5: ENSEMBLE ---")

pred_ensemble_opt = (
    0.25 * all_preds_opt['XGBoost'] +
    0.20 * all_preds_opt['Random Forest'] +
    0.05 * all_preds_opt['Ridge'] +
    0.05 * all_preds_opt['GLM'] +
    0.10 * all_preds_opt['SVR'] +
    0.15 * pred_lstm_opt +
    0.05 * pred_rnn_opt +
    0.15 * pred_gru_opt
)

final_preds_opt = {**all_preds_opt,
                   'LSTM': pred_lstm_opt,
                   'RNN': pred_rnn_opt,
                   'GRU': pred_gru_opt,
                   'Ensemble': pred_ensemble_opt}

# --- SCHRITT 6: HEATMAP ---
print("\n--- HEATMAP ---")

test_ids = groups.iloc[test_idx].values
test_months = merged_final.iloc[test_idx]['visit_month'].values

df_comp_opt = pd.DataFrame({'patient_id': test_ids, 'visit_month': test_months})
y_test_reset = y_test.reset_index(drop=True)

for col in target_cols:
    df_comp_opt[f'True_{col}'] = y_test_reset[col]

for model_name, preds in final_preds_opt.items():
    df_comp_opt = pd.concat([df_comp_opt,
        pd.DataFrame(preds, columns=[f'{model_name}_{c}' for c in target_cols])], axis=1)

metrics = []
for m, preds in final_preds_opt.items():
    for i, col in enumerate(target_cols):
        metrics.append({'Model': m, 'Target': col,
                        'MAE': mean_absolute_error(y_test_reset[col], preds[:, i])})

pivot = pd.DataFrame(metrics).pivot(index='Model', columns='Target', values='MAE')

plt.figure(figsize=(18, 10))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='viridis', linewidths=.5)
plt.title('Performance mit optimierten Features & Modellen')
plt.tight_layout()
plt.savefig('heatmap_optimized_performance.png', dpi=300)
plt.show()

In [ ]:
# Zelle 18: Visualisierung der optimierten Modelle

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# --- 1. PATIENTENSPEZIFISCHER VORHERSAGEGRAPH (AKTUALISIERT) ---
def plot_patient_opt(patient_id, df_data):
    p_data = df_data[df_data['patient_id'] == patient_id]
    if len(p_data) == 0:
        print(f"Patient {patient_id} wurde nicht gefunden.")
        return
    visit_row = p_data.iloc[0]
    models = [
        'Ist-Wert', # "True" -> Ist-Wert
        'XGBoost', 'Random Forest', 'Ridge', 'GLM', 'SVR',
        'LSTM', 'RNN', 'GRU', 'Ensemble'
    ]

    horizons = [0, 6, 12, 24]

    fig, axes = plt.subplots(2, 2, figsize=(24, 14))
    axes = axes.flatten()

    for i, updrs in enumerate([1, 2, 3, 4]):
        ax = axes[i]
        plot_data = {'Vorhersagehorizont': [], 'Punktzahl': [], 'Modell': []}

        for h in horizons:
            col = f'updrs_{updrs}_plus_{h}'
            # Wir behalten die Spaltennamen-Logik bei, ändern aber die Anzeige-Labels
            if f'True_{col}' in visit_row:
                plot_data['Vorhersagehorizont'].append(f'+{h} Mo.')
                plot_data['Punktzahl'].append(visit_row[f'True_{col}'])
                plot_data['Modell'].append('Ist-Wert')

            for m in models[1:]:
                # Mapping zurück zu den englischen Spaltennamen falls nötig
                if f'{m}_{col}' in visit_row:
                    val = visit_row[f'{m}_{col}']
                    if not pd.isna(val):
                        plot_data['Vorhersagehorizont'].append(f'+{h} Mo.')
                        plot_data['Punktzahl'].append(val)
                        plot_data['Modell'].append(m)

        if plot_data['Punktzahl']:
            sns.barplot(data=pd.DataFrame(plot_data), x='Vorhersagehorizont', y='Punktzahl', hue='Modell', ax=ax, palette='bright', alpha=0.9)
            ax.set_title(f'UPDRS Teil {updrs} Vorhersagen', fontsize=14, fontweight='bold')
            ax.set_ylabel('Score (Punktzahl)')
            ax.set_xlabel('Zeitraum')
            ax.grid(True, axis='y', linestyle='--', alpha=0.3)

            if i == 0:
                ax.legend(loc='lower left', bbox_to_anchor=(0, 1.10, 1, 0.2), mode="expand", borderaxespad=0, ncol=6)
            else:
                if ax.get_legend(): ax.get_legend().remove()

    plt.suptitle(f'Analyse der optimierten Modelle - Patient {patient_id}', fontsize=18, y=1.05)
    plt.tight_layout()

    filename = f'patient_{patient_id}_analyse_optimiert.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Gespeichert: {filename}")

    plt.show()


# --- 2. MERKMALSRELEVANZ (FEATURE IMPORTANCE) ---
def plot_imp_opt(models, features, top_n=15):
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))

    # XGBoost Wichtigkeit
    if 'XGBoost' in models:
        xgb_imp = pd.DataFrame({'Merkmal': features, 'Wichtigkeit': models['XGBoost'].feature_importances_})
        sns.barplot(data=xgb_imp.sort_values('Wichtigkeit', ascending=False).head(top_n), x='Wichtigkeit', y='Merkmal', ax=axes[0], palette='viridis')
        axes[0].set_title('XGBoost Merkmalsrelevanz (Optimiertes Set)')
        axes[0].set_xlabel('Bedeutung')
        axes[0].set_ylabel('Merkmal')

    # Random Forest Wichtigkeit
    if 'Random Forest' in models:
        rf_imp = pd.DataFrame({'Merkmal': features, 'Wichtigkeit': models['Random Forest'].feature_importances_})
        sns.barplot(data=rf_imp.sort_values('Wichtigkeit', ascending=False).head(top_n), x='Wichtigkeit', y='Merkmal', ax=axes[1], palette='magma')
        axes[1].set_title('Random Forest Merkmalsrelevanz (Optimiertes Set)')
        axes[1].set_xlabel('Bedeutung')
        axes[1].set_ylabel('Merkmal')

    plt.tight_layout()

    filename = 'merkmalsrelevanz_optimiert.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"GESPEICHERT: {filename}")

    plt.show()


# --- 3. AUSFÜHRUNG ---
print("--- VISUALISIERUNG DER OPTIMIERTEN ERGEBNISSE (VOLLSTÄNDIGE MODELLLISTE) ---")

# A. PATIENTEN-DIAGRAMM
if 'df_comp_opt' in globals():
    if 51708 in df_comp_opt['patient_id'].values:
        p_id = 51708
    else:
        p_id = np.random.choice(df_comp_opt['patient_id'].unique())

    print(f"Erstelle Plot für Patienten-ID: {p_id}")
    plot_patient_opt(p_id, df_comp_opt)
else:
    print("Fehler: 'df_comp_opt' wurde nicht gefunden.")

# B. MERKMALSRELEVANZ
if 'xgb_opt' in globals() and 'rf_opt' in globals():
    current_models_opt = {'XGBoost': xgb_opt, 'Random Forest': rf_opt}
    plot_imp_opt(current_models_opt, selected_features, top_n=15)
else:
    print("Fehler: Optimierte Modelle wurden nicht gefunden.")

In [ ]:
# =============================================================================
# Zusatzzelle: Datensatzstatistiken für das Methodikkapitel
# =============================================================================

print("--- DATENSATZSTATISTIKEN FÜR DAS METHODIKKAPITEL ---")

total_obs = len(merged_final)
total_patients = merged_final["patient_id"].nunique()

train_obs = len(X_train)
train_patients = groups.iloc[train_idx].nunique()

test_obs = len(X_test)
test_patients = groups.iloc[test_idx].nunique()

num_full_features = len(features)
num_opt_features = (
    len(selected_features)
    if "selected_features" in globals()
    else "Noch nicht berechnet; Zelle zur Feature-Selektion muss zuerst ausgeführt werden."
)

print("\n1. GESAMTDATENSATZ")
print(f"   -> Gesamte Beobachtungen: {total_obs}")
print(f"   -> Einzigartige Patienten: {total_patients}")

print("\n2. MERKMALE")
print(f"   -> Vollständiges Feature-Set: {num_full_features} Merkmale")
print(f"   -> Optimiertes Feature-Set: {num_opt_features} Merkmale")

print("\n3. TRAININGSDATENSATZ")
print(f"   -> Beobachtungen im Training: {train_obs} ({round(train_obs / total_obs * 100, 1)} %)")
print(f"   -> Einzigartige Patienten im Training: {train_patients} ({round(train_patients / total_patients * 100, 1)} %)")

print("\n4. TESTDATENSATZ")
print(f"   -> Beobachtungen im Test: {test_obs} ({round(test_obs / total_obs * 100, 1)} %)")
print(f"   -> Einzigartige Patienten im Test: {test_patients} ({round(test_patients / total_patients * 100, 1)} %)")

overlap_patients = set(groups.iloc[train_idx]).intersection(set(groups.iloc[test_idx]))

print("\n5. KONTROLLE AUF DATA LEAKAGE")
print(f"   -> Patientenüberschneidung zwischen Trainings- und Testdatensatz: {len(overlap_patients)}")

if len(overlap_patients) == 0:
    print("   Keine Überschneidung: GroupKFold hat die Patienten korrekt getrennt.")
else:
    print("   Warnung: Es wurde eine Patientenüberschneidung zwischen Training und Test gefunden.")

In [ ]:
# =============================================================================
# ZUSATZ-ZELLE 1: Training mit optimiertem Feature-Set und originalen Parametern
# =============================================================================
print("--- OPTIMIERTES SET: TRAINING MIT ORIGINALPARAMETERN STARTET ---")

all_preds_opt_same = {}

# 1. Ridge, GLM und SVR (Standardparameter)
ridge_opt_same = MultiOutputRegressor(RidgeCV(alphas=[0.1, 1.0, 10.0]))
ridge_opt_same.fit(X_train_opt, y_train)
all_preds_opt_same['Ridge'] = ridge_opt_same.predict(X_test_opt)

glm_opt_same = MultiOutputRegressor(TweedieRegressor(power=1.5, alpha=0.5, max_iter=1000))
glm_opt_same.fit(X_train_opt, y_train)
all_preds_opt_same['GLM'] = glm_opt_same.predict(X_test_opt)

svr_opt_same_pipe = Pipeline([
    ('pca', PCA(n_components=min(30, X_train_opt.shape[1]))),
    ('svr', SVR(kernel='rbf', C=1.0, epsilon=0.1))
])
svr_opt_same = MultiOutputRegressor(svr_opt_same_pipe, n_jobs=1)
svr_opt_same.fit(X_train_opt, y_train)
all_preds_opt_same['SVR'] = svr_opt_same.predict(X_test_opt)

# 2. XGBoost und Random Forest (mit best_xgb_params aus dem Full-Feature-Set)
xgb_opt_same = xgb.XGBRegressor(**best_xgb_params)  # Originale best_xgb_params werden verwendet
xgb_opt_same.fit(X_train_opt, y_train)
all_preds_opt_same['XGBoost'] = xgb_opt_same.predict(X_test_opt)

rf_opt_same = RandomForestRegressor(n_estimators=150, max_depth=12, n_jobs=-1, random_state=42)
rf_opt_same.fit(X_train_opt, y_train)
all_preds_opt_same['Random Forest'] = rf_opt_same.predict(X_test_opt)

# 3. Deep-Learning-Modelle (LSTM, RNN, GRU – originale Architektur)
print("Deep-Learning-Modelle werden trainiert...")
# Hinweis: Es wird angenommen, dass die Funktion build_dl_model zuvor definiert wurde.
lstm_opt_same = build_dl_model('LSTM', (seq_len, X_train_opt.shape[1]), y_train.shape[1])
lstm_opt_same.fit(X_train_seq_opt, y_train_seq_scaled, epochs=50, batch_size=64, verbose=0)
pred_lstm_opt_same = y_scaler_opt.inverse_transform(lstm_opt_same.predict(X_test_seq_opt))

rnn_opt_same = build_dl_model('RNN', (seq_len, X_train_opt.shape[1]), y_train.shape[1])
rnn_opt_same.fit(X_train_seq_opt, y_train_seq_scaled, epochs=50, batch_size=64, verbose=0)
pred_rnn_opt_same = y_scaler_opt.inverse_transform(rnn_opt_same.predict(X_test_seq_opt))

gru_opt_same = build_dl_model('GRU', (seq_len, X_train_opt.shape[1]), y_train.shape[1])
gru_opt_same.fit(X_train_seq_opt, y_train_seq_scaled, epochs=50, batch_size=64, verbose=0)
pred_gru_opt_same = y_scaler_opt.inverse_transform(gru_opt_same.predict(X_test_seq_opt))

# 4. Ensemble (mit originalen Gewichten)
pred_ensemble_opt_same = (
    0.25 * all_preds_opt_same['XGBoost'] + 0.20 * all_preds_opt_same['Random Forest'] +
    0.05 * all_preds_opt_same['Ridge'] + 0.05 * all_preds_opt_same['GLM'] + 0.10 * all_preds_opt_same['SVR'] +
    0.15 * pred_lstm_opt_same + 0.05 * pred_rnn_opt_same + 0.15 * pred_gru_opt_same
)

all_preds_opt_same.update({
    'LSTM': pred_lstm_opt_same,
    'RNN': pred_rnn_opt_same,
    'GRU': pred_gru_opt_same,
    'Ensemble': pred_ensemble_opt_same
})

# 5. Erstellung der Vergleichstabelle und Heatmap
df_comp_opt_same = pd.DataFrame({'patient_id': test_ids, 'visit_month': test_months})
y_test_reset = y_test.reset_index(drop=True)

for col in target_cols:
    df_comp_opt_same[f'True_{col}'] = y_test_reset[col]

metrics_opt_same = []
for m, preds in all_preds_opt_same.items():
    for i, col in enumerate(target_cols):
        mae = mean_absolute_error(y_test_reset[col], preds[:, i])
        metrics_opt_same.append({'Model': m, 'Target': col, 'MAE': mae})
        df_comp_opt_same[f'{m}_{col}'] = preds[:, i]

pivot_opt_same = pd.DataFrame(metrics_opt_same).pivot(index='Model', columns='Target', values='MAE')

plt.figure(figsize=(18, 10))
sns.heatmap(pivot_opt_same, annot=True, fmt='.2f', cmap='viridis',linewidths=.5)
plt.title('Performance-Heatmap: Optimierte Features mit Parametern des Full-Feature-Sets')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# ZUSATZ-ZELLE 2: Analyse der Vorhersagen für einen Beispielpatienten (51708)
# =============================================================================
target_patient = 51708

if target_patient in df_comp_opt_same['patient_id'].values:
    print(f"Vorhersage für Patient {target_patient} mit originalen Parametern und optimiertem Feature-Set wird visualisiert...")
    # Hinweis: Die Funktion plot_patient_opt wurde zuvor im Notebook definiert und kann direkt aufgerufen werden.
    plot_patient_opt(target_patient, df_comp_opt_same)
else:
    print(f"Fehler: Patient mit der ID {target_patient} ist im Testdatensatz nicht vorhanden.")

In [ ]:
# Cell 19: Explainable AI (XAI) - SHAP Values Integration
import shap
import matplotlib.pyplot as plt
import numpy as np

print("--- SCHRITT 7: EXPLAINABLE AI (SHAP) ANALYSE ---")

# Vorbereitung der SHAP-Bibliothek für unser XGBoost-Modell
# Hinweis: Da es sich um ein Multi-Output-Problem handelt, verwenden wir TreeExplainer
explainer = shap.TreeExplainer(xgb_opt)
shap_values = explainer(X_test_opt)

# 1. GLOBALE ERKLÄRUNG (Allgemeine Übersicht über alle Patienten)
# Welche Merkmale beeinflussen den Krankheitsverlauf insgesamt am stärksten?
print("\nErstelle globalen SHAP Summary Plot...")
plt.figure(figsize=(12, 8))

# Da es sich um ein Multi-Target (16 Targets) Problem handelt, können die SHAP-Werte eine 3D-Matrix sein.
# Wir wählen das wichtigste Ziel: "updrs_3_plus_24" (Motorik nach 2 Jahren)
# Zielindex bestimmen:
target_name = 'updrs_3_plus_24'
target_idx = list(target_cols).index(target_name)

# Nur die SHAP-Werte dieses Ziels visualisieren
shap.summary_plot(shap_values[:, :, target_idx], X_test_opt, show=False)
plt.title(f"Globale Feature-Wichtigkeit für {target_name} (SHAP Summary)", fontsize=16)
plt.tight_layout()
plt.savefig('shap_summary_global.png', dpi=300, bbox_inches='tight')
print("GESPEICHERT: shap_summary_global.png")
plt.show()

# 2. LOKALE ERKLÄRUNG (Detaillierte Analyse eines einzelnen Patienten)
# Der „personalisierte“ Teil, den ein Arzt im Dashboard sehen würde!
patient_id_to_explain = 4923
patient_indices = np.where(test_ids == patient_id_to_explain)[0]

if len(patient_indices) > 0:
    # Erste Visite dieses Patienten im Testset auswählen
    idx = patient_indices[0]
    visit_m = test_months[idx]
    print(f"\nErstelle lokale Erklärung für Patient {patient_id_to_explain} (Besuchsmonat: {visit_m})")

    plt.figure(figsize=(10, 6))
    # Waterfall-Plot zeigt, welche Faktoren den Score erhöhen oder senken
    shap.plots.waterfall(shap_values[idx, :, target_idx], show=False)
    plt.title(f"Personalisierte SHAP-Analyse - Patient {patient_id_to_explain} ({target_name})", fontsize=14)
    plt.tight_layout()
    plt.savefig(f'shap_local_patient_{patient_id_to_explain}.png', dpi=300, bbox_inches='tight')
    print(f"GESPEICHERT: shap_local_patient_{patient_id_to_explain}.png")
    plt.show()
else:
    print(f"Patient {patient_id_to_explain} wurde im Testdatensatz nicht gefunden. Bitte andere ID wählen.")

print("--- EXPLAINABLE AI (SHAP): VOLLSTÄNDIGES FEATURE-SET (1200+ FEATURES) ---")

# Dieses Mal verwenden wir das ursprüngliche Modell (mit allen Features) und den entsprechenden Testdatensatz
explainer_full = shap.TreeExplainer(xgb_model)
shap_values_full = explainer_full(X_test)

# Ziel bleibt gleich (Motorik nach 24 Monaten) für direkte Vergleichbarkeit
target_name = 'updrs_3_plus_24'
target_idx = list(target_cols).index(target_name)

# 1. GLOBALE ERKLÄRUNG (Full Set)
print("\nErstelle globalen SHAP Summary Plot für vollständiges Feature-Set...")
plt.figure(figsize=(12, 8))
# Mit max_display verhindern wir eine überladene Darstellung
shap.summary_plot(shap_values_full[:, :, target_idx], X_test, max_display=15, show=False)
plt.title(f"Globale Feature-Wichtigkeit (Vollständiges Set, 1200+) - {target_name}", fontsize=16)
plt.tight_layout()
plt.savefig('shap_summary_global_FULL.png', dpi=300, bbox_inches='tight')
print("GESPEICHERT: shap_summary_global_FULL.png")
plt.show()

# 2. LOKALE ERKLÄRUNG (Full Set - Patient 4923)
patient_id_to_explain = 4923
patient_indices = np.where(test_ids == patient_id_to_explain)[0]

if len(patient_indices) > 0:
    idx = patient_indices[0]
    visit_m = test_months[idx]
    print(f"\nErstelle lokale Erklärung für Patient {patient_id_to_explain} (Vollständiges Set, Besuchsmonat: {visit_m})")

    plt.figure(figsize=(10, 6))
    # Waterfall-Plot
    shap.plots.waterfall(shap_values_full[idx, :, target_idx], max_display=10, show=False)
    plt.title(f"Personalisierte SHAP-Analyse (Vollständiges Set) - Patient {patient_id_to_explain} ({target_name})", fontsize=14)
    plt.tight_layout()
    plt.savefig(f'shap_local_patient_{patient_id_to_explain}_FULL.png', dpi=300, bbox_inches='tight')
    print(f"GESPEICHERT: shap_local_patient_{patient_id_to_explain}_FULL.png")
    plt.show()
else:
    print(f"Patient {patient_id_to_explain} wurde im Testdatensatz nicht gefunden. Bitte andere ID wählen.")

save_dir = "results"  # Ordnername (kann beliebig angepasst werden)

# Ordner erstellen, falls nicht vorhanden
os.makedirs(save_dir, exist_ok=True)

csv_path = f'{save_dir}/optimized_predictions_comparison.csv'
df_comp_opt.to_csv(csv_path, index=False)

print(f"✅ Vergleichs-CSV wurde gespeichert: {csv_path}")

In [ ]:
# =============================================================================
# Zelle 20: TOP-5 FEATURE-TABELLE — Für jedes Target (16 Targets × 2 Modelle)
# shap_values      → Optimiertes Modell (selected_features)
# shap_values_full → Vollständiges Modell (alle ~1200 Features)
# =============================================================================

print("--- ZELLE 20: TOP-5 SHAP FEATURE-TABELLE ---")

def build_top5_table(sv, feature_names, target_names, label=""):
    """
    sv            : SHAP Explanation Objekt — Form (n_samples, n_features, n_targets)
    feature_names : Liste, Länge = n_features
    target_names  : Liste, Länge = n_targets (target_cols)
    label         : Bezeichnung für den Tabellentitel
    """
    sv_arr = sv.values if hasattr(sv, 'values') else np.array(sv)
    # sv_arr Form: (n_samples, n_features, n_targets)

    rows = []
    for t_idx, t_name in enumerate(target_names):
        mean_abs = np.abs(sv_arr[:, :, t_idx]).mean(axis=0)   # (n_features,)
        top5_idx = np.argsort(mean_abs)[::-1][:5]
        for rank, f_idx in enumerate(top5_idx, 1):
            rows.append({
                'Target'      : t_name,
                'Rank'        : rank,
                'Feature'     : feature_names[f_idx],
                'Mean |SHAP|' : round(mean_abs[f_idx], 5)
            })

    df = pd.DataFrame(rows)

    # ── Breitformat: jedes Target als Spaltengruppe ───────────────────────────
    wide = (df.pivot_table(
                index='Rank',
                columns='Target',
                values='Feature',
                aggfunc='first')
              .reindex(columns=target_names))   # Reihenfolge der Targets beibehalten

    print(f"\n{'='*60}")
    print(f"  {label} — Top-5 Features pro Target")
    print(f"{'='*60}")
    print(wide.to_string())

    return df, wide

# ── 1. Optimiertes Modell ─────────────────────────────────────────────────────
df_top5_opt, wide_opt = build_top5_table(
    shap_values, selected_features, target_cols, label="OPTIMIERTES MODELL"
)

# ── 2. Vollständiges Modell ───────────────────────────────────────────────────
df_top5_full, wide_full = build_top5_table(
    shap_values_full, features, target_cols, label="VOLLSTÄNDIGES MODELL (~1200 Features)"
)

# ── CSV speichern ─────────────────────────────────────────────────────────────
df_top5_opt.to_csv('shap_top5_per_target_optimized.csv',  index=False)
df_top5_full.to_csv('shap_top5_per_target_full.csv',      index=False)
wide_opt.to_csv('shap_top5_wide_optimized.csv')
wide_full.to_csv('shap_top5_wide_full.csv')

print("\n✅ Gespeichert: shap_top5_per_target_optimized.csv")
print("✅ Gespeichert: shap_top5_per_target_full.csv")
print("✅ Gespeichert: shap_top5_wide_optimized.csv  (Pivot-Tabelle)")
print("✅ Gespeichert: shap_top5_wide_full.csv       (Pivot-Tabelle)")

# ── Heatmap: Mean |SHAP| — Top-20 Feature × 16 Targets ────────────────────────
def plot_shap_heatmap(sv, feature_names, target_names, top_n, title, fname):
    sv_arr = sv.values if hasattr(sv, 'values') else np.array(sv)
    mean_abs_matrix = np.abs(sv_arr).mean(axis=0)          # (n_features, n_targets)

    # Global wichtigste top_n Features über alle Targets
    global_imp  = mean_abs_matrix.mean(axis=1)
    top_idx     = np.argsort(global_imp)[::-1][:top_n]
    top_feats   = [feature_names[i] for i in top_idx]

    heat_data   = pd.DataFrame(
        mean_abs_matrix[top_idx, :],
        index=top_feats,
        columns=target_names
    )

    fig, ax = plt.subplots(figsize=(20, top_n * 0.5 + 2))
    sns.heatmap(
        heat_data, annot=True, fmt='.3f', cmap='YlOrRd',
        linewidths=.4, ax=ax, cbar_kws={'label': 'Mean |SHAP|'}
    )
    ax.set_title(title, fontsize=15, fontweight='bold', pad=14)
    ax.set_xlabel('Target (UPDRS × Zeithorizont)', fontsize=11)
    ax.set_ylabel('Feature', fontsize=11)

    plt.tight_layout()
    plt.savefig(fname, dpi=300, bbox_inches='tight')
    print(f"✅ GESPEICHERT: {fname}")
    plt.show()


plot_shap_heatmap(
    shap_values, selected_features, target_cols,
    top_n=20,
    title="Optimiertes Modell — Top-20 Features × 16 Targets (Mean |SHAP|)",
    fname="shap_heatmap_top20_optimized.png"
)

plot_shap_heatmap(
    shap_values_full, features, target_cols,
    top_n=20,
    title="Vollständiges Modell — Top-20 Features × 16 Targets (Mean |SHAP|)",
    fname="shap_heatmap_top20_full.png"
)

print("\n✅ Zelle 20 erfolgreich abgeschlossen.")

In [ ]:
# =============================================================================
# Zelle 21: TOP-5 BIOMARKER PRO TARGET (SHAP — Nur Bio-Features)
# Lag-/Trend-/klinische Features werden herausgefiltert, sodass nur
# Peptid-/Protein-SHAP-Werte analysiert werden.
# Für jedes der 16 Targets werden die 5 wichtigsten Biomarker bestimmt.
# =============================================================================

print("--- ZELLE 21: TOP-5 BIOMARKER PRO TARGET (SHAP) ---")

import shap, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Welche Indizes gehören zu Bio-Features? ───────────────────────────────────
clinical_kw  = ['lag', 'trend', 'roll', 'visit_month', 'medication']

# Optimiertes Modell
bio_idx_opt  = [i for i, f in enumerate(selected_features)
                if not any(k in f for k in clinical_kw)]
bio_feats_opt = [selected_features[i] for i in bio_idx_opt]

# Vollständiges Modell
bio_idx_full  = [i for i, f in enumerate(features)
                 if not any(k in f for k in clinical_kw)]
bio_feats_full = [features[i] for i in bio_idx_full]

print(f"Optimiert: {len(bio_feats_opt)} Bio-Features")
print(f"Voll:      {len(bio_feats_full)} Bio-Features")


def bio_top5_table(sv, bio_indices, bio_names, target_names):
    """
    sv          : SHAP Explanation — Form (n_samples, n_features, n_targets)
    bio_indices : Indizes der Bio-Features im ursprünglichen Feature-Raum
    bio_names   : Namen der Bio-Features
    target_names: Namen der 16 Targets
    Rückgabe    : Long-Format DataFrame und Mean-|SHAP|-Matrix (bio_feats × targets)
    """
    sv_arr   = sv.values if hasattr(sv, 'values') else np.array(sv)
    bio_sv   = sv_arr[:, bio_indices, :]              # (n_samples, n_bio, n_targets)
    mean_abs = np.abs(bio_sv).mean(axis=0)            # (n_bio, n_targets)

    rows = []
    for t_idx, t_name in enumerate(target_names):
        col_imp  = mean_abs[:, t_idx]
        top5_idx = np.argsort(col_imp)[::-1][:5]
        for rank, f_idx in enumerate(top5_idx, 1):
            rows.append({
                'Target'     : t_name,
                'Rank'       : rank,
                'Biomarker'  : bio_names[f_idx],
                'Mean |SHAP|': round(col_imp[f_idx], 6)
            })

    df_long = pd.DataFrame(rows)

    mat = pd.DataFrame(
        mean_abs,
        index   = bio_names,
        columns = target_names
    )
    return df_long, mat


def plot_bio_heatmap(mat, top_n, title, fname):
    """
    mat   : DataFrame (bio_feats × targets) — Mean |SHAP|
    top_n : Zeigt global die wichtigsten N Bio-Features
    """
    global_imp = mat.mean(axis=1).sort_values(ascending=False)
    top_feats  = global_imp.head(top_n).index
    heat       = mat.loc[top_feats]

    fig, ax = plt.subplots(figsize=(22, top_n * 0.52 + 2.5))
    sns.heatmap(
        heat, annot=True, fmt='.4f', cmap='YlOrRd',
        linewidths=.4, ax=ax,
        cbar_kws={'label': 'Mean |SHAP|', 'shrink': 0.6}
    )
    ax.set_title(title, fontsize=14, fontweight='bold', pad=14)
    ax.set_xlabel('Target (UPDRS × Zeithorizont)', fontsize=11)
    ax.set_ylabel('Biomarker', fontsize=11)
    ax.tick_params(axis='y', labelsize=9)
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.tight_layout()
    plt.savefig(fname, dpi=300, bbox_inches='tight')
    print(f"✅ GESPEICHERT: {fname}")
    plt.show()


def print_top5_table(df_long, label):
    wide = (df_long.pivot_table(
                index='Rank', columns='Target',
                values='Biomarker', aggfunc='first')
              .reindex(columns=target_cols))
    print(f"\n{'='*70}")
    print(f"  {label} — Top-5 Biomarker pro Target")
    print(f"{'='*70}")
    print(wide.to_string())


# ── Optimiertes Modell ────────────────────────────────────────────────────────
df_opt, mat_opt = bio_top5_table(
    shap_values, bio_idx_opt, bio_feats_opt, target_cols
)
print_top5_table(df_opt, "OPTIMIERTES MODELL")
plot_bio_heatmap(
    mat_opt, top_n=10,
    title="Optimiertes Modell — Top-10 Biomarker × 16 Targets (Mean |SHAP|)",
    fname="shap_bio_heatmap_optimized.png"
)
df_opt.to_csv('shap_bio_top5_optimized.csv', index=False)
print("✅ shap_bio_top5_optimized.csv gespeichert")

# ── Vollständiges Modell ──────────────────────────────────────────────────────
df_full, mat_full = bio_top5_table(
    shap_values_full, bio_idx_full, bio_feats_full, target_cols
)
print_top5_table(df_full, "VOLLSTÄNDIGES MODELL (~1200 Features)")
plot_bio_heatmap(
    mat_full, top_n=10,
    title="Vollständiges Modell — Top-10 Biomarker × 16 Targets (Mean |SHAP|)",
    fname="shap_bio_heatmap_full.png"
)
df_full.to_csv('shap_bio_top5_full.csv', index=False)
print("✅ shap_bio_top5_full.csv gespeichert")

print("\n✅ Zelle 21 erfolgreich abgeschlossen.")

In [ ]:
# =============================================================================
# Zelle 22: Gerichtete SHAP-Werte im vollständigen Modell
# =============================================================================

print("--- ZELLE 22: GERICHTETE SHAP-WERTE IM VOLLSTÄNDIGEN MODELL ---")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

clinical_keywords = ["lag", "trend", "roll", "visit_month", "medication"]

bio_idx_full = [
    i for i, feature in enumerate(features)
    if not any(keyword in feature for keyword in clinical_keywords)
]

bio_feats_full = [features[i] for i in bio_idx_full]

sv_arr = (
    shap_values_full.values
    if hasattr(shap_values_full, "values")
    else np.array(shap_values_full)
)

bio_sv = sv_arr[:, bio_idx_full, :]

# Gerichteter Mittelwert:
# Positive Werte erhöhen die Modellprognose, negative Werte senken die Modellprognose.
signed_mean = bio_sv.mean(axis=0)

signed_mat = pd.DataFrame(
    signed_mean,
    index=bio_feats_full,
    columns=target_cols
)

rows_signed = []

for t_idx, target_name in enumerate(target_cols):
    col = signed_mean[:, t_idx]

    positive_indices = np.argsort(col)[::-1][:5]
    for rank, f_idx in enumerate(positive_indices, 1):
        rows_signed.append({
            "Zielvariable": target_name,
            "Rang": rank,
            "Biomarker": bio_feats_full[f_idx],
            "Mean_SHAP": round(col[f_idx], 6),
            "Richtung": "Prognose_erhöhend" if col[f_idx] > 0 else "Prognose_senkend"
        })

    negative_indices = np.argsort(col)[:5]
    for rank, f_idx in enumerate(negative_indices, 1):
        rows_signed.append({
            "Zielvariable": target_name,
            "Rang": -rank,
            "Biomarker": bio_feats_full[f_idx],
            "Mean_SHAP": round(col[f_idx], 6),
            "Richtung": "Prognose_senkend" if col[f_idx] < 0 else "Prognose_erhöhend"
        })

df_signed = pd.DataFrame(rows_signed)

signed_csv_path = TABLES_DIR / "shap_bio_gerichtet_full.csv"
df_signed.to_csv(signed_csv_path, index=False)

print(f"Gerichtete SHAP-Tabelle gespeichert unter: {signed_csv_path}")

union_feats = df_signed["Biomarker"].unique()
heat = signed_mat.loc[signed_mat.index.isin(union_feats)].copy()
heat = heat.loc[heat.abs().mean(axis=1).sort_values(ascending=False).index]

vmax = heat.abs().values.max()

fig, ax = plt.subplots(figsize=(24, len(heat) * 0.55 + 2.5))

sns.heatmap(
    heat,
    annot=True,
    fmt=".3f",
    cmap="RdBu_r",
    center=0,
    vmin=-vmax,
    vmax=vmax,
    linewidths=.4,
    ax=ax,
    cbar_kws={
        "label": "Mean SHAP: positiv = prognoseerhöhend, negativ = prognosesenkend",
        "shrink": 0.5
    }
)

ax.set_title(
    "Vollständiges Modell — gerichtete Biomarker-SHAP-Werte pro Zielvariable\n"
    "(Rot = prognoseerhöhend, Blau = prognosesenkend)",
    fontsize=13,
    fontweight="bold",
    pad=14
)

ax.set_xlabel("Zielvariable (UPDRS × Vorhersagehorizont)", fontsize=11)
ax.set_ylabel("Biomarker", fontsize=11)
ax.tick_params(axis="y", labelsize=8)

plt.xticks(rotation=45, ha="right", fontsize=9)
plt.tight_layout()

signed_heatmap_path = FIGURES_DIR / "shap_bio_gerichtete_heatmap_full.png"
plt.savefig(signed_heatmap_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Gerichtete SHAP-Heatmap gespeichert unter: {signed_heatmap_path}")

In [ ]:
# =============================================================================
# Zelle 23: Export der Modellartefakte für das Dashboard
# =============================================================================

print("--- ZELLE 24: EXPORT DER MODELLARTEFAKTE FÜR DAS DASHBOARD ---")

# Artefakte des optimierten Modells
joblib.dump(xgb_opt, DASHBOARD_ASSETS_DIR / "xgb_model_optimized.pkl")
joblib.dump(y_scaler_opt, DASHBOARD_ASSETS_DIR / "y_scaler.pkl")
joblib.dump(selected_features, DASHBOARD_ASSETS_DIR / "selected_features.pkl")

test_data_optimized = {
    "X_test": X_test_opt,
    "predictions": df_comp_opt,
    "patient_ids": test_ids,
    "visit_months": test_months
}

joblib.dump(test_data_optimized, DASHBOARD_ASSETS_DIR / "test_data.pkl")

print(f"Artefakte des optimierten Modells gespeichert unter: {DASHBOARD_ASSETS_DIR}")


# Artefakte des vollständigen Modells
joblib.dump(xgb_model, DASHBOARD_ASSETS_FULL_DIR / "xgb_model_full.pkl")
joblib.dump(features, DASHBOARD_ASSETS_FULL_DIR / "all_features.pkl")

full_test_data = {
    "X_test": X_test,
    "predictions": df_comp,
    "patient_ids": test_ids,
    "visit_months": test_months
}

joblib.dump(full_test_data, DASHBOARD_ASSETS_FULL_DIR / "test_data_full.pkl")
joblib.dump(y_scaler_opt, DASHBOARD_ASSETS_FULL_DIR / "y_scaler.pkl")

print(f"Artefakte des vollständigen Modells gespeichert unter: {DASHBOARD_ASSETS_FULL_DIR}")


# Peptid-UniProt-Mapping
print("Erstelle Peptid-UniProt-Mapping aus den Rohdaten...")

train_pep = pd.read_csv(
    DATA_FILES["train_peptides"],
    usecols=["Peptide", "UniProt"],
    dtype={"Peptide": str, "UniProt": str}
)

peptide_map = (
    train_pep
    .drop_duplicates("Peptide")
    .set_index("Peptide")["UniProt"]
    .to_dict()
)

joblib.dump(peptide_map, DASHBOARD_ASSETS_DIR / "peptide_map.pkl")
joblib.dump(peptide_map, DASHBOARD_ASSETS_FULL_DIR / "peptide_map.pkl")

print(f"Anzahl einzigartiger Peptide im Mapping: {len(peptide_map)}")
print("Export der Dashboard-Artefakte abgeschlossen.")

In [ ]:
print("--- ZELLE 24: EXPORT DES FULL-FEATURE-MODELLS ---")
print(f"Anzahl der Features: {len(features)} (klinisch + alle Peptid/Protein + Lag/Trend/Roll)")

save_dir_full = 'dashboard_assets_full'
os.makedirs(save_dir_full, exist_ok=True)

# Modell: xgb_model aus Zelle 8 (mit allen Features trainiert)
joblib.dump(xgb_model, f'{save_dir_full}/xgb_model_full.pkl')
print("✅ Gespeichert: xgb_model_full.pkl")

# Feature-Liste (alle ~1200 Features)
joblib.dump(features, f'{save_dir_full}/all_features.pkl')
print(f"✅ Gespeichert: all_features.pkl ({len(features)} Features)")

# Testdaten — Verwendung von df_comp aus Zelle 14 (original, korrigiert)
full_test_data = {
    'X_test':       X_test,          # Testset mit allen Features
    'predictions':  df_comp,         # Vergleichstabelle aus Zelle 14
    'patient_ids':  test_ids,
    'visit_months': test_months
}
joblib.dump(full_test_data, f'{save_dir_full}/test_data_full.pkl')
print("✅ Gespeichert: test_data_full.pkl")

# Scaler (für DL-Konsistenz gespeichert, wird in der App nicht verwendet)
joblib.dump(y_scaler_opt, f'{save_dir_full}/y_scaler.pkl')
print("✅ Gespeichert: y_scaler.pkl")

print(f"\n📦 Im Ordner dashboard_assets_full/ sind {len(os.listdir(save_dir_full))} Dateien bereit.")

In [ ]:
# =============================================================================
# Zelle 25: Peptid zu UniProt Mapping (Wörterbuch)
# =============================================================================
# Erstellung eines Mappings zwischen Peptiden und Proteinen für die Darstellung.
print("Lese train_peptides.csv (ca. 900k Zeilen)...")
train_pep = pd.read_csv(base_path + 'train_peptides.csv', usecols=['Peptide', 'UniProt'], dtype={'Peptide': str, 'UniProt': str})

# Zuweisung über ein Dictionary
peptide_map = train_pep.drop_duplicates('Peptide').set_index('Peptide')['UniProt'].to_dict()

print(f"✅ Anzahl einzigartiger Peptide : {len(peptide_map)}")
for d in ['dashboard_assets', 'dashboard_assets_full']:
    if os.path.exists(d):
        joblib.dump(peptide_map, f'{d}/peptide_map.pkl')

print("\n🚀 SKRIPT ERFOLGREICH BEENDET! Alle Dateien sind exportbereit für die Thesis und das Dashboard.")

In [ ]:
# =============================================================================
# Zelle 26: Positiver und negativer SHAP-Bericht für alle Merkmale
# =============================================================================

print("--- ZELLE 23: POSITIVER UND NEGATIVER SHAP-BERICHT FÜR ALLE MERKMALE ---")

import numpy as np
import pandas as pd

if "peptide_map" not in globals():
    print("Hinweis: peptide_map wurde noch nicht gefunden. Für Proteinmerkmale wird der Merkmalsname als UniProt-Code verwendet.")
    peptide_map = {}

clinical_keywords = ["lag", "trend", "roll", "visit_month", "medication"]

sv_arr_full = (
    shap_values_full.values
    if hasattr(shap_values_full, "values")
    else np.array(shap_values_full)
)

all_idx = list(range(len(features)))


def get_feature_type(feature_name):
    """Klassifiziert ein Merkmal als klinisches oder biologisches Merkmal."""
    if any(keyword in feature_name for keyword in clinical_keywords):
        return "Klinisch"
    return "Biologisch"


def get_uniprot(feature_name, feature_type):
    """Ermittelt den zugehörigen UniProt-Code, sofern ein Mapping vorhanden ist."""
    if feature_type == "Klinisch":
        return "-"
    if feature_name in peptide_map:
        return peptide_map[feature_name]
    return feature_name


def get_subscale_indices(best_target, all_targets):
    """Findet alle Zielvariablen derselben UPDRS-Subskala."""
    if not best_target:
        return []
    base_updrs = best_target.split("_plus_")[0]
    return [i for i, target in enumerate(all_targets) if target.startswith(base_updrs)]


def build_peak_table_positive(feature_indices, feature_names, shap_array, target_names):
    """Erstellt eine Tabelle der stärksten positiven SHAP-Beiträge pro Merkmal."""
    rows = []

    for f_local, f_global in enumerate(feature_indices):
        best_pos_mean = -np.inf
        best_freq = 0.0
        best_net = -np.inf
        best_target = ""

        for t_idx, target_name in enumerate(target_names):
            shap_vals = shap_array[:, f_global, t_idx]
            pos_mask = shap_vals > 0

            freq = pos_mask.mean() * 100
            pos_mean = shap_vals[pos_mask].mean() if pos_mask.sum() > 0 else 0.0
            net_mean = shap_vals.mean()

            if pos_mean > best_pos_mean:
                best_pos_mean = pos_mean
                best_freq = freq
                best_net = net_mean
                best_target = target_name

        subscale_pos_mean = 0.0

        if best_target:
            subscale_idx = get_subscale_indices(best_target, target_names)
            subscale_shap = shap_array[:, f_global, subscale_idx]
            subscale_pos_mask = subscale_shap > 0
            subscale_pos_mean = (
                subscale_shap[subscale_pos_mask].mean()
                if subscale_pos_mask.sum() > 0
                else 0.0
            )

        feature_name = feature_names[f_local]
        feature_type = get_feature_type(feature_name)

        rows.append({
            "Merkmal": feature_name,
            "Datentyp": feature_type,
            "Peak_Target": best_target,
            "Peak_Target_Haeufigkeit_%": round(best_freq, 1),
            "Peak_Target_Mittelwert_Positive_SHAP": round(best_pos_mean, 5) if best_pos_mean != -np.inf else 0.0,
            "UPDRS_Subskala_Mittelwert_Positive_SHAP": round(subscale_pos_mean, 5),
            "Peak_Target_Netto_Mittelwert_SHAP": round(best_net, 5) if best_net != -np.inf else 0.0
        })

    return pd.DataFrame(rows)


def build_peak_table_negative(feature_indices, feature_names, shap_array, target_names):
    """Erstellt eine Tabelle der stärksten negativen SHAP-Beiträge pro Merkmal."""
    rows = []

    for f_local, f_global in enumerate(feature_indices):
        best_neg_mean = np.inf
        best_freq = 0.0
        best_net = np.inf
        best_target = ""

        for t_idx, target_name in enumerate(target_names):
            shap_vals = shap_array[:, f_global, t_idx]
            neg_mask = shap_vals < 0

            freq = neg_mask.mean() * 100
            neg_mean = shap_vals[neg_mask].mean() if neg_mask.sum() > 0 else 0.0
            net_mean = shap_vals.mean()

            if neg_mean < best_neg_mean:
                best_neg_mean = neg_mean
                best_freq = freq
                best_net = net_mean
                best_target = target_name

        subscale_neg_mean = 0.0

        if best_target:
            subscale_idx = get_subscale_indices(best_target, target_names)
            subscale_shap = shap_array[:, f_global, subscale_idx]
            subscale_neg_mask = subscale_shap < 0
            subscale_neg_mean = (
                subscale_shap[subscale_neg_mask].mean()
                if subscale_neg_mask.sum() > 0
                else 0.0
            )

        feature_name = feature_names[f_local]
        feature_type = get_feature_type(feature_name)

        rows.append({
            "Merkmal": feature_name,
            "Datentyp": feature_type,
            "Peak_Target": best_target,
            "Peak_Target_Haeufigkeit_%": round(best_freq, 1),
            "Peak_Target_Mittelwert_Negative_SHAP": round(best_neg_mean, 5) if best_neg_mean != np.inf else 0.0,
            "UPDRS_Subskala_Mittelwert_Negative_SHAP": round(subscale_neg_mean, 5),
            "Peak_Target_Netto_Mittelwert_SHAP": round(best_net, 5) if best_net != np.inf else 0.0
        })

    return pd.DataFrame(rows)


print("Berechne positive SHAP-Beiträge...")
all_df_pos = build_peak_table_positive(all_idx, features, sv_arr_full, target_cols)

print("Berechne negative SHAP-Beiträge...")
all_df_neg = build_peak_table_negative(all_idx, features, sv_arr_full, target_cols)

all_pos_final = all_df_pos[all_df_pos["Peak_Target_Netto_Mittelwert_SHAP"] > 0].copy()
all_neg_final = all_df_neg[all_df_neg["Peak_Target_Netto_Mittelwert_SHAP"] < 0].copy()

all_pos_final.insert(
    2,
    "UniProt_Code",
    all_pos_final.apply(
        lambda row: get_uniprot(row["Merkmal"], row["Datentyp"]),
        axis=1
    )
)

all_neg_final.insert(
    2,
    "UniProt_Code",
    all_neg_final.apply(
        lambda row: get_uniprot(row["Merkmal"], row["Datentyp"]),
        axis=1
    )
)

excel_path = TABLES_DIR / "shap_all_features_positive_negative_rankings_FINAL.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    all_pos_final.sort_values(
        "Peak_Target_Mittelwert_Positive_SHAP",
        ascending=False
    ).to_excel(
        writer,
        sheet_name="Positiv_Mittelwert",
        index=False
    )

    all_pos_final.sort_values(
        "Peak_Target_Haeufigkeit_%",
        ascending=False
    ).to_excel(
        writer,
        sheet_name="Positiv_Haeufigkeit",
        index=False
    )

    all_neg_final.sort_values(
        "Peak_Target_Mittelwert_Negative_SHAP",
        ascending=True
    ).to_excel(
        writer,
        sheet_name="Negativ_Mittelwert",
        index=False
    )

    all_neg_final.sort_values(
        "Peak_Target_Haeufigkeit_%",
        ascending=False
    ).to_excel(
        writer,
        sheet_name="Negativ_Haeufigkeit",
        index=False
    )

print(f"Excel-Bericht gespeichert unter: {excel_path}")
print(f"Anzahl positiver Merkmale mit Netto-SHAP > 0: {len(all_pos_final)}")
print(f"Anzahl negativer Merkmale mit Netto-SHAP < 0: {len(all_neg_final)}")